<a href="https://colab.research.google.com/github/madalamanikanta/ImageCaptioning_MiniProject/blob/manikanta-dev/07_CLIP_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 07 — CLIP + LSTM Image Captioning
Complete training notebook using the already extracted CLIP features and tokenized captions.
No dataset extraction or CLIP extraction is repeated. The notebook loads the 120 feature chunks,
keeps the 119,865 × 512 feature matrix in RAM (~234 MB), uses 0-based image indices, packed LSTM
sequences, mixed precision, AdamW, validation, checkpointing, and resume support.

## Step 1 — Connect Google Drive

In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive")
except ValueError as e:
    if "Mountpoint must not already contain files" in str(e):
        print("Drive is already mounted; continuing.")
    else:
        raise
print("Drive setup complete.")

Mounted at /content/drive
Drive setup complete.


## Step 2 — Imports and Project Paths

In [ ]:
from pathlib import Path
import os, gc, json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from tqdm.auto import tqdm

PROJECT_DIR = Path("/content/drive/MyDrive/ImageCaptioning_MiniProject")
PROCESSED_DIR = PROJECT_DIR / "Processed"
CLIP_DIR = PROJECT_DIR / "Features" / "CLIP"
MODEL_DIR = PROJECT_DIR / "Models" / "CLIP_LSTM"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_DIR.exists())
print("Processed exists:", PROCESSED_DIR.exists())
print("CLIP directory exists:", CLIP_DIR.exists())

Project exists: True
Processed exists: True
CLIP directory exists: True


## Step 3 — GPU Check

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*60)
print("GPU CONFIGURATION")
print("="*60)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")
else:
    raise RuntimeError("GPU is not available. Do not start this training on CPU.")
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
print("Device:", DEVICE)

GPU CONFIGURATION
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8
GPU memory: 14.56 GB
Device: cuda


## Step 4 — Load Vocabulary

In [ ]:
with open(PROCESSED_DIR / "vocabulary.json", "r", encoding="utf-8") as f:
    vocabulary_data = json.load(f)

word_to_id = vocabulary_data["word_to_id"]
id_to_word = vocabulary_data["id_to_word"]
VOCAB_SIZE = int(vocabulary_data["vocab_size"])
MAX_LENGTH = int(vocabulary_data["max_length"])

PAD_ID = int(word_to_id["<PAD>"])
UNK_ID = int(word_to_id["<UNK>"])
START_ID = int(word_to_id["<start>"])
END_ID = int(word_to_id["<end>"])

print("Vocabulary size:", VOCAB_SIZE)
print("Maximum length:", MAX_LENGTH)
print("<PAD>:", PAD_ID, "<UNK>:", UNK_ID, "<start>:", START_ID, "<end>:", END_ID)

assert (VOCAB_SIZE, MAX_LENGTH) == (21649, 80)
assert (PAD_ID, UNK_ID, START_ID, END_ID) == (0,1,2,3)
print("✓ Vocabulary verified.")

Vocabulary size: 21649
Maximum length: 80
<PAD>: 0 <UNK>: 1 <start>: 2 <end>: 3
✓ Vocabulary verified.


## Step 5 — Load Tokenized Captions and Image Indices

In [ ]:
train_captions = np.load(PROCESSED_DIR / "train_caption_sequences.npy", mmap_mode="r")
val_captions = np.load(PROCESSED_DIR / "val_caption_sequences.npy", mmap_mode="r")
test_captions = np.load(PROCESSED_DIR / "test_caption_sequences.npy", mmap_mode="r")
train_indices = np.load(PROCESSED_DIR / "train_image_indices.npy", mmap_mode="r")
val_indices = np.load(PROCESSED_DIR / "val_image_indices.npy", mmap_mode="r")
test_indices = np.load(PROCESSED_DIR / "test_image_indices.npy", mmap_mode="r")

print("Train:", train_captions.shape, train_indices.shape)
print("Val  :", val_captions.shape, val_indices.shape)
print("Test :", test_captions.shape, test_indices.shape)

assert train_captions.shape == (479416,80)
assert val_captions.shape == (59923,80)
assert test_captions.shape == (59939,80)
assert len(train_captions)==len(train_indices)
assert len(val_captions)==len(val_indices)
assert len(test_captions)==len(test_indices)
print("✓ Tokenized data verified.")

Train: (479416, 80) (479416,)
Val  : (59923, 80) (59923,)
Test : (59939, 80) (59939,)
✓ Tokenized data verified.


## Step 6 — Verify 120 CLIP Feature Chunks

In [ ]:
feature_files = sorted(CLIP_DIR.glob("features_*.npy"))
print("Feature files:", len(feature_files))
assert len(feature_files) == 120

total = 0
for f in feature_files:
    a = np.load(f, mmap_mode="r")
    assert a.shape[1] == 512
    total += a.shape[0]

assert total == 119865
print("Total feature vectors:", total)
print("✓ CLIP features verified.")

Feature files: 120
Total feature vectors: 119865
✓ CLIP features verified.


## Step 7 — Load All CLIP Features into RAM

In [ ]:
print("Loading CLIP features into RAM...")
arrays = []
for i, f in enumerate(feature_files):
    arrays.append(np.load(f))
    if (i+1) % 20 == 0:
        print(f"Loaded {i+1}/120")

clip_feature_matrix = np.concatenate(arrays, axis=0).astype(np.float32, copy=False)
del arrays
gc.collect()

print("Shape:", clip_feature_matrix.shape)
print("Dtype:", clip_feature_matrix.dtype)
print("RAM size:", round(clip_feature_matrix.nbytes/1024**2, 2), "MB")
assert clip_feature_matrix.shape == (119865,512)
print("✓ All CLIP features loaded.")

Loading CLIP features into RAM...
Loaded 20/120
Loaded 40/120
Loaded 60/120
Loaded 80/120
Loaded 100/120
Loaded 120/120
Shape: (119865, 512)
Dtype: float32
RAM size: 234.11 MB
✓ All CLIP features loaded.


## Step 8 — Dataset and Fast RAM Collator

In [ ]:
class ImageCaptionDataset(Dataset):
    def __init__(self, captions, image_indices):
        self.captions = captions
        self.image_indices = image_indices
    def __len__(self):
        return len(self.captions)
    def __getitem__(self, idx):
        return np.asarray(self.captions[idx], dtype=np.int64), int(self.image_indices[idx])

class FastCLIPCaptionCollator:
    def __call__(self, batch):
        captions = np.stack([x[0] for x in batch]).astype(np.int64, copy=False)
        image_indices = np.asarray([x[1] for x in batch], dtype=np.int64)
        # IMPORTANT: indices are already 0-based. Do not subtract 1.
        features = clip_feature_matrix[image_indices]
        return torch.from_numpy(features), torch.from_numpy(captions)

train_dataset = ImageCaptionDataset(train_captions, train_indices)
val_dataset = ImageCaptionDataset(val_captions, val_indices)
test_dataset = ImageCaptionDataset(test_captions, test_indices)
collator = FastCLIPCaptionCollator()

print(len(train_dataset), len(val_dataset), len(test_dataset))

479416 59923 59939


## Step 9 — Fast DataLoaders

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, collate_fn=collator)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collator)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 7491
Val batches: 937
Test batches: 937


## Step 10 — Exact Feature Alignment Test

In [ ]:
alignment_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=collator)
ram_features, _ = next(iter(alignment_loader))
expected = clip_feature_matrix[train_indices[:8]]
difference = np.max(np.abs(expected - ram_features.numpy()))
print("Expected:", expected.shape)
print("Loaded:", ram_features.shape)
print("Max difference:", difference)
assert difference < 1e-6
print("✓ Feature alignment verified.")

Expected: (8, 512)
Loaded: torch.Size([8, 512])
Max difference: 0.0
✓ Feature alignment verified.


## Step 11 — Define CLIP + LSTM

In [ ]:
class CLIPLSTM(nn.Module):
    def __init__(self, clip_dim=512, vocab_size=21649, embed_dim=512, hidden_dim=512, num_layers=2, dropout=0.30):
        super().__init__()
        self.num_layers = num_layers
        self.image_projection = nn.Sequential(nn.Linear(clip_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.Tanh())
        self.image_cell_projection = nn.Sequential(nn.Linear(clip_dim, hidden_dim), nn.Tanh())
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, image_features, captions, lengths=None):
        inputs = captions[:, :-1]
        h0 = self.image_projection(image_features).unsqueeze(0).repeat(self.num_layers,1,1)
        c0 = self.image_cell_projection(image_features).unsqueeze(0).repeat(self.num_layers,1,1)
        embeddings = self.dropout(self.embedding(inputs))

        if lengths is not None:
            input_lengths = (lengths - 1).clamp(min=1)
            packed = pack_padded_sequence(embeddings, input_lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_output, _ = self.lstm(packed, (h0,c0))
            lstm_output, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=inputs.size(1))
        else:
            lstm_output, _ = self.lstm(embeddings, (h0,c0))

        return self.output_layer(self.dropout(lstm_output))

MODEL_CONFIG = dict(clip_dim=512, vocab_size=VOCAB_SIZE, embed_dim=512, hidden_dim=512, num_layers=2, dropout=0.30)
model = CLIPLSTM(**MODEL_CONFIG).to(DEVICE)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

Parameters: 26,919,057


## Step 12 — Loss, Optimizer, Scheduler and AMP

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1, min_lr=1e-6)
scaler = torch.amp.GradScaler("cuda")
GRADIENT_CLIP = 1.0

print("✓ Training configuration ready.")

✓ Training configuration ready.


## Step 13 — One-Batch Sanity Test

In [ ]:
model.train()
features, captions = next(iter(train_loader))
features = features.to(DEVICE, non_blocking=True)
captions = captions.to(DEVICE, non_blocking=True)
lengths = captions.ne(PAD_ID).sum(dim=1)
targets = captions[:,1:]

optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(features, captions, lengths)
    loss = criterion(logits.reshape(-1,VOCAB_SIZE), targets.reshape(-1))
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
scaler.step(optimizer)
scaler.update()

print("Logits:", logits.shape)
print("Loss:", float(loss))
print("Gradient norm:", float(grad_norm))
print("Average length:", round(lengths.float().mean().item(),2))
assert logits.shape == (BATCH_SIZE, MAX_LENGTH-1, VOCAB_SIZE)
print("✓ One-batch training test passed.")

Logits: torch.Size([64, 79, 21649])
Loss: 9.978025436401367
Gradient norm: 0.21885886788368225
Average length: 13.45
✓ One-batch training test passed.


/tmp/ipykernel_1649/2273110526.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Loss:", float(loss))


## Step 14 — Training and Validation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0.0
    total_batches = 0
    progress = tqdm(loader, desc="Training", leave=False)

    for features, captions in progress:
        features = features.to(device, non_blocking=True)
        captions = captions.to(device, non_blocking=True)
        lengths = captions.ne(PAD_ID).sum(dim=1)
        targets = captions[:,1:]
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(features, captions, lengths)
            loss = criterion(logits.reshape(-1,VOCAB_SIZE), targets.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()

        v = float(loss.detach())
        total_loss += v
        total_batches += 1
        progress.set_postfix(loss=f"{v:.4f}")

    return total_loss / max(total_batches,1)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    progress = tqdm(loader, desc="Validation", leave=False)

    for features, captions in progress:
        features = features.to(device, non_blocking=True)
        captions = captions.to(device, non_blocking=True)
        lengths = captions.ne(PAD_ID).sum(dim=1)
        targets = captions[:,1:]

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(features, captions, lengths)
            loss = criterion(logits.reshape(-1,VOCAB_SIZE), targets.reshape(-1))

        v = float(loss)
        total_loss += v
        total_batches += 1
        progress.set_postfix(loss=f"{v:.4f}")

    return total_loss / max(total_batches,1)

## Step 15 — Checkpoint and Resume Settings

In [ ]:
BEST_MODEL_PATH = MODEL_DIR / "best_model.pth"
LAST_MODEL_PATH = MODEL_DIR / "last_model.pth"
HISTORY_PATH = MODEL_DIR / "training_history.json"

EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
CHECKPOINT_EVERY_BATCHES = 1000

def save_checkpoint(path, epoch, batch_in_epoch, best_val_loss, history):
    torch.save({
        "epoch": epoch,
        "batch_in_epoch": batch_in_epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
        "model_config": MODEL_CONFIG
    }, path)

RESUME = True
start_epoch = 0
resume_batch = 0
best_val_loss = float("inf")
epochs_without_improvement = 0
history = {"train_loss": [], "val_loss": [], "learning_rate": []}

if RESUME and LAST_MODEL_PATH.exists():
    ckpt = torch.load(LAST_MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = int(ckpt["epoch"])
    resume_batch = int(ckpt.get("batch_in_epoch",0))
    best_val_loss = float(ckpt["best_val_loss"])
    history = ckpt["history"]
    print("✓ Resumed checkpoint:", LAST_MODEL_PATH)
    print("Epoch:", start_epoch, "Batch:", resume_batch)
else:
    print("No checkpoint found. Starting fresh.")

No checkpoint found. Starting fresh.


## Step 16 — Full Training

In [ ]:
for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    print("\n" + "="*70)
    print(f"EPOCH {epoch+1}/{EPOCHS}")
    print("="*70)

    model.train()
    total_loss = 0.0
    total_batches = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1} Training")

    for batch_number, (features, captions) in enumerate(progress):
        if epoch == start_epoch and batch_number < resume_batch:
            continue

        features = features.to(DEVICE, non_blocking=True)
        captions = captions.to(DEVICE, non_blocking=True)
        lengths = captions.ne(PAD_ID).sum(dim=1)
        targets = captions[:,1:]
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(features, captions, lengths)
            loss = criterion(logits.reshape(-1,VOCAB_SIZE), targets.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()

        v = float(loss.detach())
        total_loss += v
        total_batches += 1
        progress.set_postfix(loss=f"{v:.4f}")

        if (batch_number+1) % CHECKPOINT_EVERY_BATCHES == 0:
            save_checkpoint(LAST_MODEL_PATH, epoch, batch_number+1, best_val_loss, history)
            print(f"\n✓ Resume checkpoint saved at batch {batch_number+1}")

    resume_batch = 0
    train_loss = total_loss / max(total_batches,1)

    print("\nRunning validation...")
    val_loss = validate_one_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["learning_rate"].append(float(lr))

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"LR:         {lr:.7f}")
    print(f"Epoch time: {(time.time()-epoch_start)/60:.2f} min")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        save_checkpoint(BEST_MODEL_PATH, epoch+1, 0, best_val_loss, history)
        print("✓ NEW BEST MODEL:", best_val_loss)
    else:
        epochs_without_improvement += 1
        print("No improvement:", epochs_without_improvement, "/", EARLY_STOPPING_PATIENCE)

    save_checkpoint(LAST_MODEL_PATH, epoch+1, 0, best_val_loss, history)
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTraining finished.")
print("Best validation loss:", best_val_loss)


EPOCH 1/10


Epoch 1 Training:   0%|          | 0/7491 [00:00<?, ?it/s]


✓ Resume checkpoint saved at batch 1000

✓ Resume checkpoint saved at batch 2000

✓ Resume checkpoint saved at batch 3000

✓ Resume checkpoint saved at batch 4000

✓ Resume checkpoint saved at batch 5000

✓ Resume checkpoint saved at batch 6000

✓ Resume checkpoint saved at batch 7000

Running validation...


Validation:   0%|          | 0/937 [00:00<?, ?it/s]

Train Loss: 3.8923
Val Loss:   3.3223
LR:         0.0003000
Epoch time: 9.46 min
✓ NEW BEST MODEL: 3.3222797685397216

EPOCH 2/10


...

Train Loss: 2.9149
Val Loss:   2.9508
LR:         0.0003000
Epoch time: 9.18 min
✓ NEW BEST MODEL: 2.950808772664187

EPOCH 10/10


Epoch 10 Training:   0%|          | 0/7491 [00:00<?, ?it/s]


✓ Resume checkpoint saved at batch 1000

✓ Resume checkpoint saved at batch 2000

✓ Resume checkpoint saved at batch 3000

✓ Resume checkpoint saved at batch 4000

✓ Resume checkpoint saved at batch 5000

✓ Resume checkpoint saved at batch 6000

✓ Resume checkpoint saved at batch 7000

Running validation...


Validation:   0%|          | 0/937 [00:00<?, ?it/s]

Train Loss: 2.8920
Val Loss:   2.9448
LR:         0.0003000
Epoch time: 9.19 min
✓ NEW BEST MODEL: 2.9448060823988125

Training finished.
Best validation loss: 2.9448060823988125


## Step 17 — Training Summary

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print("Best validation loss:", best_val_loss)
print("Best model:", BEST_MODEL_PATH)
print("Last checkpoint:", LAST_MODEL_PATH)
print("History:", HISTORY_PATH)
for i,(tr,va,lr) in enumerate(zip(history["train_loss"], history["val_loss"], history["learning_rate"]),1):
    print(f"Epoch {i:02d} | Train {tr:.4f} | Val {va:.4f} | LR {lr:.7f}")

TRAINING SUMMARY
Best validation loss: 2.9448060823988125
Best model: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM/best_model.pth
Last checkpoint: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM/last_model.pth
History: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM/training_history.json
Epoch 01 | Train 3.8923 | Val 3.3223 | LR 0.0003000
Epoch 02 | Train 3.3305 | Val 3.1526 | LR 0.0003000
Epoch 03 | Train 3.1915 | Val 3.0739 | LR 0.0003000
Epoch 04 | Train 3.1091 | Val 3.0280 | LR 0.0003000
Epoch 05 | Train 3.0513 | Val 3.0011 | LR 0.0003000
Epoch 06 | Train 3.0063 | Val 2.9810 | LR 0.0003000
Epoch 07 | Train 2.9707 | Val 2.9683 | LR 0.0003000
Epoch 08 | Train 2.9407 | Val 2.9585 | LR 0.0003000
Epoch 09 | Train 2.9149 | Val 2.9508 | LR 0.0003000
Epoch 10 | Train 2.8920 | Val 2.9448 | LR 0.0003000


In [ ]:
# ============================================================
# STEP 1 — RUNTIME + GOOGLE DRIVE SETUP
# ============================================================

import os
import gc
import json
import time
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive

# Mount Drive
try:
    drive.mount("/content/drive")
except ValueError as e:
    if "already contain files" in str(e):
        print("Drive already mounted — continuing.")
    else:
        raise

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ImageCaptioning_MiniProject"
)

PROCESSED_DIR = PROJECT_DIR / "Processed"
CLIP_DIR = PROJECT_DIR / "Features" / "CLIP"

MODEL_DIR = PROJECT_DIR / "Models" / "CLIP_LSTM_Optimized"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("RUNTIME SETUP")
print("=" * 60)

print("Project exists :", PROJECT_DIR.exists())
print("Processed      :", PROCESSED_DIR.exists())
print("CLIP directory :", CLIP_DIR.exists())
print("Model directory:", MODEL_DIR)

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print()
print("Device :", DEVICE)

if DEVICE.type != "cuda":
    raise RuntimeError(
        "GPU is NOT available. Stop here and reconnect to a GPU runtime."
    )

print("GPU    :", torch.cuda.get_device_name(0))
print(
    "VRAM   :",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

# Performance settings
torch.backends.cudnn.benchmark = True

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

gc.collect()
torch.cuda.empty_cache()

print()
print("✓ Runtime ready")

Mounted at /content/drive
RUNTIME SETUP
Project exists : True
Processed      : True
CLIP directory : True
Model directory: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized

Device : cuda
GPU    : Tesla T4
VRAM   : 14.56 GB

✓ Runtime ready


In [ ]:
# ============================================================
# STEP 2 — LOAD VERIFIED VOCABULARY
# ============================================================

VOCAB_PATH = PROCESSED_DIR / "vocabulary.json"

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocabulary_data = json.load(f)

word_to_id = vocabulary_data["word_to_id"]
id_to_word = vocabulary_data["id_to_word"]

VOCAB_SIZE = int(vocabulary_data["vocab_size"])
MAX_LENGTH = int(vocabulary_data["max_length"])

PAD_ID = int(word_to_id["<PAD>"])
UNK_ID = int(word_to_id["<UNK>"])
START_ID = int(word_to_id["<start>"])
END_ID = int(word_to_id["<end>"])

print("=" * 60)
print("VOCABULARY LOADED")
print("=" * 60)

print("Vocabulary size :", VOCAB_SIZE)
print("Maximum length  :", MAX_LENGTH)

print()
print("Special tokens:")
print("<PAD>   :", PAD_ID)
print("<UNK>   :", UNK_ID)
print("<start> :", START_ID)
print("<end>   :", END_ID)

# Safety verification
assert VOCAB_SIZE == 21649
assert MAX_LENGTH == 80

assert PAD_ID == 0
assert UNK_ID == 1
assert START_ID == 2
assert END_ID == 3

print()
print("✓ Vocabulary verified")

VOCABULARY LOADED
Vocabulary size : 21649
Maximum length  : 80

Special tokens:
<PAD>   : 0
<UNK>   : 1
<start> : 2
<end>   : 3

✓ Vocabulary verified


In [ ]:
# ============================================================
# STEP 3 — LOAD TOKENIZED CAPTIONS + IMAGE INDICES
# ============================================================

train_captions = np.load(
    PROCESSED_DIR / "train_caption_sequences.npy",
    mmap_mode="r"
)

val_captions = np.load(
    PROCESSED_DIR / "val_caption_sequences.npy",
    mmap_mode="r"
)

test_captions = np.load(
    PROCESSED_DIR / "test_caption_sequences.npy",
    mmap_mode="r"
)

train_indices = np.load(
    PROCESSED_DIR / "train_image_indices.npy",
    mmap_mode="r"
)

val_indices = np.load(
    PROCESSED_DIR / "val_image_indices.npy",
    mmap_mode="r"
)

test_indices = np.load(
    PROCESSED_DIR / "test_image_indices.npy",
    mmap_mode="r"
)

print("=" * 60)
print("TOKENIZED DATA LOADED")
print("=" * 60)

print("Train captions :", train_captions.shape)
print("Train indices  :", train_indices.shape)

print("Validation captions :", val_captions.shape)
print("Validation indices  :", val_indices.shape)

print("Test captions :", test_captions.shape)
print("Test indices  :", test_indices.shape)

# ------------------------------------------------------------
# VERIFICATION
# ------------------------------------------------------------

assert train_captions.shape == (479416, 80)
assert val_captions.shape == (59923, 80)
assert test_captions.shape == (59939, 80)

assert train_indices.shape == (479416,)
assert val_indices.shape == (59923,)
assert test_indices.shape == (59939,)

assert len(train_captions) == len(train_indices)
assert len(val_captions) == len(val_indices)
assert len(test_captions) == len(test_indices)

print()
print("✓ Train aligned")
print("✓ Validation aligned")
print("✓ Test aligned")
print("✓ Tokenized dataset verified")

TOKENIZED DATA LOADED
Train captions : (479416, 80)
Train indices  : (479416,)
Validation captions : (59923, 80)
Validation indices  : (59923,)
Test captions : (59939, 80)
Test indices  : (59939,)

✓ Train aligned
✓ Validation aligned
✓ Test aligned
✓ Tokenized dataset verified


In [ ]:
# ============================================================
# STEP 4 — LOAD CLIP FEATURES INTO RAM
# ============================================================

feature_files = sorted(
    CLIP_DIR.glob("features_*.npy")
)

print("=" * 60)
print("CLIP FEATURE FILES")
print("=" * 60)

print("Feature files found:", len(feature_files))

assert len(feature_files) == 120, (
    f"Expected 120 feature files, found {len(feature_files)}"
)

print("First:", feature_files[0].name)
print("Last :", feature_files[-1].name)

# ------------------------------------------------------------
# LOAD ALL FEATURES
# ------------------------------------------------------------

feature_arrays = []

for i, feature_file in enumerate(feature_files):

    features = np.load(feature_file)

    assert features.ndim == 2
    assert features.shape[1] == 512

    feature_arrays.append(features)

    if (i + 1) % 20 == 0:
        print(f"Loaded {i + 1}/120 files")

# Combine all chunks
clip_feature_matrix = np.concatenate(
    feature_arrays,
    axis=0
).astype(np.float32, copy=False)

del feature_arrays
gc.collect()

print()
print("=" * 60)
print("CLIP FEATURES LOADED")
print("=" * 60)

print("Shape :", clip_feature_matrix.shape)
print("Dtype :", clip_feature_matrix.dtype)

print(
    "RAM size:",
    round(
        clip_feature_matrix.nbytes / (1024 ** 2),
        2
    ),
    "MB"
)

# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

assert clip_feature_matrix.shape == (
    119865,
    512
)

assert clip_feature_matrix.dtype == np.float32

print()
print("✓ 120 chunks loaded")
print("✓ 119,865 feature vectors")
print("✓ Every feature is 512-dimensional")
print("✓ Feature matrix ready in RAM")

CLIP FEATURE FILES
Feature files found: 120
First: features_0000.npy
Last : features_0119.npy
Loaded 20/120 files
Loaded 40/120 files
Loaded 60/120 files
Loaded 80/120 files
Loaded 100/120 files
Loaded 120/120 files

CLIP FEATURES LOADED
Shape : (119865, 512)
Dtype : float32
RAM size: 234.11 MB

✓ 120 chunks loaded
✓ 119,865 feature vectors
✓ Every feature is 512-dimensional
✓ Feature matrix ready in RAM


In [ ]:
# ============================================================
# STEP 5 — FAST DATASET + DATALOADER
# ============================================================

class ImageCaptionDataset(Dataset):

    def __init__(self, captions, image_indices):
        self.captions = captions
        self.image_indices = image_indices

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):

        caption = np.asarray(
            self.captions[idx],
            dtype=np.int64
        )

        image_index = int(
            self.image_indices[idx]
        )

        return caption, image_index


class FastCLIPCaptionCollator:

    def __call__(self, batch):

        captions = np.stack(
            [item[0] for item in batch]
        ).astype(
            np.int64,
            copy=False
        )

        image_indices = np.asarray(
            [item[1] for item in batch],
            dtype=np.int64
        )

        # IMPORTANT:
        # Image indices are already 0-based.
        # Do NOT subtract 1.
        features = clip_feature_matrix[
            image_indices
        ]

        return (
            torch.from_numpy(features),
            torch.from_numpy(captions)
        )


# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------

train_dataset = ImageCaptionDataset(
    train_captions,
    train_indices
)

val_dataset = ImageCaptionDataset(
    val_captions,
    val_indices
)

test_dataset = ImageCaptionDataset(
    test_captions,
    test_indices
)


# ------------------------------------------------------------
# COLLATOR
# ------------------------------------------------------------

fast_collator = FastCLIPCaptionCollator()


# ------------------------------------------------------------
# DATALOADERS
# ------------------------------------------------------------

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=fast_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=fast_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=fast_collator
)


print("=" * 60)
print("FAST DATA PIPELINE READY")
print("=" * 60)

print("Train samples :", len(train_dataset))
print("Validation    :", len(val_dataset))
print("Test samples  :", len(test_dataset))

print()
print("Batch size    :", BATCH_SIZE)

print()
print("Train batches :", len(train_loader))
print("Val batches   :", len(val_loader))
print("Test batches  :", len(test_loader))

FAST DATA PIPELINE READY
Train samples : 479416
Validation    : 59923
Test samples  : 59939

Batch size    : 64

Train batches : 7491
Val batches   : 937
Test batches  : 937


In [ ]:
# ============================================================
# STEP 6 — FEATURE ALIGNMENT VERIFICATION
# ============================================================

alignment_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=fast_collator
)

ram_features, ram_captions = next(
    iter(alignment_loader)
)

expected_indices = train_indices[:8]

expected_features = clip_feature_matrix[
    expected_indices
]

difference = np.max(
    np.abs(
        expected_features
        - ram_features.numpy()
    )
)

print("=" * 60)
print("FEATURE ALIGNMENT TEST")
print("=" * 60)

print("Expected shape :", expected_features.shape)
print("Loaded shape   :", ram_features.shape)
print("Max difference :", difference)

assert difference < 1e-6

print()
print("✓ Feature alignment = 0.0")
print("✓ RAM loader matches original CLIP features")


FEATURE ALIGNMENT TEST
Expected shape : (8, 512)
Loaded shape   : torch.Size([8, 512])
Max difference : 0.0

✓ Feature alignment = 0.0
✓ RAM loader matches original CLIP features


In [ ]:
# ============================================================
# STEP 8 — OPTIMIZED CLIP + LSTM
# ============================================================

from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)


class OptimizedCLIPLSTM(nn.Module):

    def __init__(
        self,
        clip_dim=512,
        vocab_size=21649,
        embed_dim=512,
        hidden_dim=512,
        num_layers=2,
        dropout=0.30
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # ----------------------------------------------------
        # CLIP FEATURE NORMALIZATION
        # ----------------------------------------------------

        self.clip_norm = nn.LayerNorm(
            clip_dim
        )

        # ----------------------------------------------------
        # CLIP → INITIAL HIDDEN STATE
        # ----------------------------------------------------

        self.image_projection = nn.Sequential(
            nn.Linear(
                clip_dim,
                hidden_dim
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        # ----------------------------------------------------
        # CLIP → INITIAL CELL STATE
        # ----------------------------------------------------

        self.cell_projection = nn.Sequential(
            nn.Linear(
                clip_dim,
                hidden_dim
            ),
            nn.Tanh()
        )

        # ----------------------------------------------------
        # WORD EMBEDDING
        # ----------------------------------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PAD_ID
        )

        # ----------------------------------------------------
        # LSTM DECODER
        # ----------------------------------------------------

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        self.dropout = nn.Dropout(
            dropout
        )

        self.output_layer = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(
        self,
        image_features,
        captions,
        lengths
    ):

        # ----------------------------------------------------
        # Caption input
        # ----------------------------------------------------

        inputs = captions[:, :-1]

        # ----------------------------------------------------
        # Normalize CLIP features
        # ----------------------------------------------------

        image_features = self.clip_norm(
            image_features
        )

        # ----------------------------------------------------
        # Initialize hidden + cell state
        # ----------------------------------------------------

        h0 = self.image_projection(
            image_features
        )

        c0 = self.cell_projection(
            image_features
        )

        h0 = h0.unsqueeze(0).repeat(
            self.num_layers,
            1,
            1
        )

        c0 = c0.unsqueeze(0).repeat(
            self.num_layers,
            1,
            1
        )

        # ----------------------------------------------------
        # Word embeddings
        # ----------------------------------------------------

        embeddings = self.embedding(
            inputs
        )

        embeddings = self.dropout(
            embeddings
        )

        # ----------------------------------------------------
        # PACKED LSTM
        # ----------------------------------------------------

        input_lengths = (
            lengths - 1
        ).clamp(min=1)

        packed = pack_padded_sequence(
            embeddings,
            input_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(
            packed,
            (h0, c0)
        )

        lstm_output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=inputs.size(1)
        )

        # ----------------------------------------------------
        # VOCABULARY PREDICTION
        # ----------------------------------------------------

        lstm_output = self.dropout(
            lstm_output
        )

        logits = self.output_layer(
            lstm_output
        )

        return logits

In [ ]:
# ============================================================
# CELL 2 — CREATE OPTIMIZED MODEL
# ============================================================

MODEL_CONFIG = {
    "clip_dim": 512,
    "vocab_size": VOCAB_SIZE,
    "embed_dim": 512,
    "hidden_dim": 512,
    "num_layers": 2,
    "dropout": 0.30
}

model = OptimizedCLIPLSTM(
    **MODEL_CONFIG
).to(DEVICE)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("OPTIMIZED CLIP + LSTM")
print("=" * 60)

print("Total parameters    :", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Device              :", DEVICE)

OPTIMIZED CLIP + LSTM
Total parameters    : 26,920,081
Trainable parameters: 26,920,081
Device              : cuda


In [ ]:
# ============================================================
# CELL 3 — FORWARD PASS TEST
# ============================================================

model.eval()

features, captions = next(
    iter(train_loader)
)

features = features.to(
    DEVICE,
    non_blocking=True
)

captions = captions.to(
    DEVICE,
    non_blocking=True
)

lengths = captions.ne(
    PAD_ID
).sum(dim=1)

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        logits = model(
            features,
            captions,
            lengths
        )

print("=" * 60)
print("FORWARD PASS")
print("=" * 60)

print("Features :", features.shape)
print("Captions :", captions.shape)
print("Lengths  :", lengths.shape)
print("Logits   :", logits.shape)

print(
    "Average caption length:",
    round(
        lengths.float().mean().item(),
        2
    )
)

assert logits.shape == (
    BATCH_SIZE,
    MAX_LENGTH - 1,
    VOCAB_SIZE
)

print()
print("✓ Optimized model forward pass successful")

FORWARD PASS
Features : torch.Size([64, 512])
Captions : torch.Size([64, 80])
Lengths  : torch.Size([64])
Logits   : torch.Size([64, 79, 21649])
Average caption length: 13.69

✓ Optimized model forward pass successful


In [ ]:
# ============================================================
# STEP 9 — LOSS + OPTIMIZER + SCHEDULER + AMP
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

scaler = torch.amp.GradScaler("cuda")

GRADIENT_CLIP = 1.0

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

print("Loss            : CrossEntropyLoss")
print("Label smoothing : 0.05")
print("Optimizer       : AdamW")
print("Learning rate   : 3e-4")
print("Weight decay    : 1e-4")
print("Scheduler       : ReduceLROnPlateau")
print("AMP             : FP16")
print("Gradient clip   :", GRADIENT_CLIP)

print()
print("✓ Training configuration ready")

TRAINING CONFIGURATION
Loss            : CrossEntropyLoss
Label smoothing : 0.05
Optimizer       : AdamW
Learning rate   : 3e-4
Weight decay    : 1e-4
Scheduler       : ReduceLROnPlateau
AMP             : FP16
Gradient clip   : 1.0

✓ Training configuration ready


In [ ]:
# ============================================================
# STEP 10 — ONE-BATCH TRAINING TEST
# ============================================================

model.train()

features, captions = next(
    iter(train_loader)
)

features = features.to(
    DEVICE,
    non_blocking=True
)

captions = captions.to(
    DEVICE,
    non_blocking=True
)

lengths = captions.ne(
    PAD_ID
).sum(dim=1)

targets = captions[:, 1:]

optimizer.zero_grad(
    set_to_none=True
)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):

    logits = model(
        features,
        captions,
        lengths
    )

    loss = criterion(
        logits.reshape(-1, VOCAB_SIZE),
        targets.reshape(-1)
    )

scaler.scale(
    loss
).backward()

scaler.unscale_(
    optimizer
)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    GRADIENT_CLIP
)

scaler.step(
    optimizer
)

scaler.update()

print("=" * 60)
print("ONE-BATCH TRAINING TEST")
print("=" * 60)

print("Feature shape :", features.shape)
print("Caption shape :", captions.shape)
print("Logits shape  :", logits.shape)
print("Loss          :", float(loss))
print("Gradient norm :", float(grad_norm))

print()
print("✓ Forward pass")
print("✓ Packed sequence")
print("✓ Loss calculation")
print("✓ Backpropagation")
print("✓ Gradient clipping")
print("✓ Optimizer update")

ONE-BATCH TRAINING TEST
Feature shape : torch.Size([64, 512])
Caption shape : torch.Size([64, 80])
Logits shape  : torch.Size([64, 79, 21649])
Loss          : 9.991726875305176
Gradient norm : 0.26850149035453796

✓ Forward pass
✓ Packed sequence
✓ Loss calculation
✓ Backpropagation
✓ Gradient clipping
✓ Optimizer update


/tmp/ipykernel_1393/1424550129.py:73: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Loss          :", float(loss))


In [ ]:
# ============================================================
# STEP 11 — FINAL OPTIMIZED CLIP + LSTM TRAINING
# ============================================================

BEST_MODEL_PATH = MODEL_DIR / "best_model.pth"
LAST_MODEL_PATH = MODEL_DIR / "last_model.pth"
HISTORY_PATH = MODEL_DIR / "training_history.json"

EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3

# Frequent enough to protect against Colab disconnects,
# but not so frequent that Drive I/O becomes a bottleneck.
CHECKPOINT_EVERY = 500

history = {
    "train_loss": [],
    "val_loss": [],
    "learning_rate": []
}

best_val_loss = float("inf")
epochs_without_improvement = 0

start_epoch = 0
resume_batch = 0


# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def save_training_checkpoint(
    path,
    epoch,
    batch_number,
    best_val_loss,
    epochs_without_improvement
):

    checkpoint = {
        "epoch": epoch,
        "batch_number": batch_number,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "best_val_loss":
            best_val_loss,

        "epochs_without_improvement":
            epochs_without_improvement,

        "history":
            history,

        "model_config":
            MODEL_CONFIG,

        "vocab_size":
            VOCAB_SIZE,

        "max_length":
            MAX_LENGTH,

        "pad_id":
            PAD_ID,

        "start_id":
            START_ID,

        "end_id":
            END_ID
    }

    torch.save(
        checkpoint,
        path
    )


# ============================================================
# RESUME IF CHECKPOINT EXISTS
# ============================================================

if LAST_MODEL_PATH.exists():

    print("=" * 70)
    print("CHECKPOINT FOUND")
    print("=" * 70)

    checkpoint = torch.load(
        LAST_MODEL_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    scaler.load_state_dict(
        checkpoint["scaler_state_dict"]
    )

    start_epoch = checkpoint["epoch"]

    resume_batch = checkpoint.get(
        "batch_number",
        0
    )

    best_val_loss = checkpoint[
        "best_val_loss"
    ]

    epochs_without_improvement = checkpoint.get(
        "epochs_without_improvement",
        0
    )

    history = checkpoint[
        "history"
    ]

    print("✓ Checkpoint loaded")
    print("Starting epoch :", start_epoch + 1)
    print("Resume batch   :", resume_batch)
    print("Best val loss  :", best_val_loss)

else:

    print("=" * 70)
    print("NO CHECKPOINT FOUND")
    print("=" * 70)

    print("Starting training from scratch.")


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(
    start_epoch,
    EPOCHS
):

    print()
    print("=" * 70)
    print(
        f"EPOCH {epoch + 1}/{EPOCHS}"
    )
    print("=" * 70)

    epoch_start = time.time()

    model.train()

    running_loss = 0.0
    batch_count = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}",
        unit="batch"
    )

    for batch_number, (
        features,
        captions
    ) in enumerate(progress):

        # ----------------------------------------------------
        # Resume support
        # ----------------------------------------------------

        if (
            epoch == start_epoch
            and batch_number < resume_batch
        ):
            continue

        # ----------------------------------------------------
        # GPU transfer
        # ----------------------------------------------------

        features = features.to(
            DEVICE,
            non_blocking=True
        )

        captions = captions.to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Caption lengths
        # ----------------------------------------------------

        lengths = captions.ne(
            PAD_ID
        ).sum(dim=1)

        targets = captions[:, 1:]

        # ----------------------------------------------------
        # Reset gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Mixed precision forward pass
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(
                features,
                captions,
                lengths
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                targets.reshape(-1)
            )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        scaler.scale(
            loss
        ).backward()

        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )

        # ----------------------------------------------------
        # Optimizer update
        # ----------------------------------------------------

        scaler.step(
            optimizer
        )

        scaler.update()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        loss_value = loss.detach().item()

        running_loss += loss_value
        batch_count += 1

        average_loss = (
            running_loss /
            batch_count
        )

        progress.set_postfix(
            loss=f"{loss_value:.4f}",
            avg=f"{average_loss:.4f}"
        )

        # ----------------------------------------------------
        # PERIODIC CHECKPOINT
        # ----------------------------------------------------

        if (
            (batch_number + 1)
            % CHECKPOINT_EVERY
            == 0
        ):

            save_training_checkpoint(
                LAST_MODEL_PATH,
                epoch,
                batch_number + 1,
                best_val_loss,
                epochs_without_improvement
            )

            print(
                f"\n✓ Checkpoint saved "
                f"at batch {batch_number + 1}"
            )

    # --------------------------------------------------------
    # Reset resume position
    # --------------------------------------------------------

    resume_batch = 0

    train_loss = (
        running_loss /
        max(batch_count, 1)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    print()
    print("Running validation...")

    model.eval()

    validation_loss = 0.0
    validation_batches = 0

    with torch.no_grad():

        validation_progress = tqdm(
            val_loader,
            desc="Validation",
            unit="batch"
        )

        for (
            features,
            captions
        ) in validation_progress:

            features = features.to(
                DEVICE,
                non_blocking=True
            )

            captions = captions.to(
                DEVICE,
                non_blocking=True
            )

            lengths = captions.ne(
                PAD_ID
            ).sum(dim=1)

            targets = captions[:, 1:]

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    features,
                    captions,
                    lengths
                )

                loss = criterion(
                    logits.reshape(
                        -1,
                        VOCAB_SIZE
                    ),
                    targets.reshape(-1)
                )

            loss_value = loss.detach().item()

            validation_loss += loss_value
            validation_batches += 1

            validation_progress.set_postfix(
                loss=f"{loss_value:.4f}"
            )

    val_loss = (
        validation_loss /
        max(validation_batches, 1)
    )

    # --------------------------------------------------------
    # Learning-rate scheduler
    # --------------------------------------------------------

    scheduler.step(
        val_loss
    )

    current_lr = optimizer.param_groups[0][
        "lr"
    ]

    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history["train_loss"].append(
        float(train_loss)
    )

    history["val_loss"].append(
        float(val_loss)
    )

    history["learning_rate"].append(
        float(current_lr)
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    # ========================================================
    # EPOCH RESULTS
    # ========================================================

    print()
    print("=" * 70)
    print(
        f"EPOCH {epoch + 1} RESULTS"
    )
    print("=" * 70)

    print(
        f"Train Loss      : "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation Loss : "
        f"{val_loss:.4f}"
    )

    print(
        f"Learning Rate   : "
        f"{current_lr:.7f}"
    )

    print(
        f"Epoch Time      : "
        f"{epoch_time / 60:.2f} minutes"
    )

    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        epochs_without_improvement = 0

        save_training_checkpoint(
            BEST_MODEL_PATH,
            epoch + 1,
            0,
            best_val_loss,
            epochs_without_improvement
        )

        print()
        print(
            "🏆 NEW BEST MODEL"
        )

        print(
            f"Best validation loss: "
            f"{best_val_loss:.4f}"
        )

    else:

        epochs_without_improvement += 1

        print()
        print(
            "No validation improvement:"
        )

        print(
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    # ========================================================
    # SAVE LAST CHECKPOINT
    # ========================================================

    save_training_checkpoint(
        LAST_MODEL_PATH,
        epoch + 1,
        0,
        best_val_loss,
        epochs_without_improvement
    )

    with open(
        HISTORY_PATH,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )

    print()
    print("✓ Last checkpoint saved")

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print()
        print("=" * 70)
        print("EARLY STOPPING")
        print("=" * 70)

        break


# ============================================================
# TRAINING COMPLETE
# ============================================================

print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Best model:",
    BEST_MODEL_PATH
)

print(
    "Last checkpoint:",
    LAST_MODEL_PATH
)

print(
    "Training history:",
    HISTORY_PATH
)

NO CHECKPOINT FOUND
Starting training from scratch.

EPOCH 1/10


Epoch 1:   0%|          | 0/7491 [00:00<?, ?batch/s]


✓ Checkpoint saved at batch 500

✓ Checkpoint saved at batch 1000

✓ Checkpoint saved at batch 1500

✓ Checkpoint saved at batch 2000

✓ Checkpoint saved at batch 2500

✓ Checkpoint saved at batch 3000

✓ Checkpoint saved at batch 3500

✓ Checkpoint saved at batch 4000

✓ Checkpoint saved at batch 4500

✓ Checkpoint saved at batch 5000

✓ Checkpoint saved at batch 5500

✓ Checkpoint saved at batch 6000

✓ Checkpoint saved at batch 6500

✓ Checkpoint saved at batch 7000

Running validation...


Validation:   0%|          | 0/937 [00:00<?, ?batch/s]


EPOCH 1 RESULTS
Train Loss      : 3.8502
Validation Loss : 3.2958
Learning Rate   : 0.0003000
Epoch Time      : 9.06 minutes

🏆 NEW BEST MODEL
Best validation loss: 3.2958

✓ Last checkpoint saved

EPOCH 2/10


...


EPOCH 9 RESULTS
Train Loss      : 2.8936
Validation Loss : 2.9498
Learning Rate   : 0.0003000
Epoch Time      : 9.24 minutes

🏆 NEW BEST MODEL
Best validation loss: 2.9498

✓ Last checkpoint saved

EPOCH 10/10


Epoch 10:   0%|          | 0/7491 [00:00<?, ?batch/s]


✓ Checkpoint saved at batch 500

✓ Checkpoint saved at batch 1000

✓ Checkpoint saved at batch 1500

✓ Checkpoint saved at batch 2000

✓ Checkpoint saved at batch 2500

✓ Checkpoint saved at batch 3000

✓ Checkpoint saved at batch 3500

✓ Checkpoint saved at batch 4000

✓ Checkpoint saved at batch 4500

✓ Checkpoint saved at batch 5000

✓ Checkpoint saved at batch 5500

✓ Checkpoint saved at batch 6000

✓ Checkpoint saved at batch 6500

✓ Checkpoint saved at batch 7000

Running validation...


Validation:   0%|          | 0/937 [00:00<?, ?batch/s]


EPOCH 10 RESULTS
Train Loss      : 2.8719
Validation Loss : 2.9457
Learning Rate   : 0.0003000
Epoch Time      : 9.13 minutes

🏆 NEW BEST MODEL
Best validation loss: 2.9457

✓ Last checkpoint saved

TRAINING COMPLETE
Best validation loss: 2.945657781755657
Best model: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/best_model.pth
Last checkpoint: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/last_model.pth
Training history: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/training_history.json


STEP 12 — Load Best Model

In [ ]:
# ============================================================
# STEP 12 — LOAD BEST CLIP + LSTM MODEL
# ============================================================

BEST_MODEL_PATH = (
    PROJECT_DIR
    / "Models"
    / "CLIP_LSTM_Optimized"
    / "best_model.pth"
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("=" * 60)
print("BEST MODEL LOADED")
print("=" * 60)

print("Checkpoint :", BEST_MODEL_PATH)
print("Epoch      :", checkpoint["epoch"])
print("Best Val Loss:", checkpoint["best_val_loss"])
print("Device     :", DEVICE)

print()
print("✓ Best Model 1 loaded")

BEST MODEL LOADED
Checkpoint : /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/best_model.pth
Epoch      : 10
Best Val Loss: 2.945657781755657
Device     : cuda

✓ Best Model 1 loaded


STEP 13 — Prepare Test Images

In [ ]:
# ============================================================
# STEP 13 — PREPARE UNIQUE TEST IMAGES
# ============================================================

test_indices_array = np.asarray(
    test_indices,
    dtype=np.int64
)

test_captions_array = np.asarray(
    test_captions,
    dtype=np.int64
)

unique_test_images = np.unique(
    test_indices_array
)

print("=" * 60)
print("TEST SET PREPARATION")
print("=" * 60)

print("Test caption rows :", len(test_captions_array))
print("Unique test images:", len(unique_test_images))

# ------------------------------------------------------------
# Group all reference captions by image index
# ------------------------------------------------------------

test_references = {}

for image_index, caption in zip(
    test_indices_array,
    test_captions_array
):

    image_index = int(image_index)

    test_references.setdefault(
        image_index,
        []
    ).append(caption)

print(
    "Reference images:",
    len(test_references)
)

# Safety check
assert set(
    unique_test_images
) == set(
    test_references.keys()
)

print()
print("✓ Test references grouped correctly")

TEST SET PREPARATION
Test caption rows : 59939
Unique test images: 11987
Reference images: 11987

✓ Test references grouped correctly


STEP 14 — Caption Decoder

In [ ]:
# ============================================================
# STEP 14 — TOKEN → WORD DECODER
# ============================================================

def decode_caption(
    token_ids,
    remove_special=True
):

    words = []

    for token_id in token_ids:

        token_id = int(token_id)

        if token_id == END_ID:
            break

        if remove_special and token_id in (
            PAD_ID,
            START_ID
        ):
            continue

        word = id_to_word.get(
            str(token_id),
            id_to_word.get(
                token_id,
                "<UNK>"
            )
        )

        if word in (
            "<PAD>",
            "<start>",
            "<end>"
        ):
            continue

        words.append(word)

    return " ".join(words)

STEP 15 — Greedy Caption Generation

In [ ]:
# ============================================================
# STEP 15 — FIXED BATCHED GREEDY CAPTION GENERATION
# ============================================================

@torch.no_grad()
def generate_captions_greedy(
    image_indices,
    batch_size=128,
    max_length=80
):

    model.eval()

    generated_captions = {}

    image_indices = np.asarray(
        image_indices,
        dtype=np.int64
    )

    for start in tqdm(
        range(
            0,
            len(image_indices),
            batch_size
        ),
        desc="Generating test captions"
    ):

        batch_indices = image_indices[
            start:start + batch_size
        ]

        features = torch.from_numpy(
            clip_feature_matrix[
                batch_indices
            ]
        ).to(
            DEVICE,
            non_blocking=True
        )

        current_batch_size = features.size(0)

        # ----------------------------------------------------
        # Start with:
        #
        # <start> <PAD> <PAD> ...
        #
        # We need TWO positions because the model receives
        # captions[:, :-1].
        # ----------------------------------------------------

        captions = torch.full(
            (
                current_batch_size,
                max_length
            ),
            PAD_ID,
            dtype=torch.long,
            device=DEVICE
        )

        captions[:, 0] = START_ID

        finished = torch.zeros(
            current_batch_size,
            dtype=torch.bool,
            device=DEVICE
        )

        # ----------------------------------------------------
        # Generate one token at a time
        # ----------------------------------------------------

        for step in range(
            1,
            max_length
        ):

            # At step=1:
            # captions = <start> <PAD>
            #
            # Model input becomes:
            # <start>
            #
            # Therefore it can safely pack a sequence
            # of length 1.

            current_captions = captions[
                :, :step + 1
            ]

            current_lengths = torch.full(
                (current_batch_size,),
                step + 1,
                dtype=torch.long,
                device=DEVICE
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    features,
                    current_captions,
                    current_lengths
                )

            # ------------------------------------------------
            # Predict the next token
            # ------------------------------------------------

            next_token = torch.argmax(
                logits[:, -1, :],
                dim=-1
            )

            # Once END was generated, keep END
            next_token = torch.where(
                finished,
                torch.full_like(
                    next_token,
                    END_ID
                ),
                next_token
            )

            captions[:, step] = next_token

            finished |= (
                next_token == END_ID
            )

            if finished.all():
                break

        # ----------------------------------------------------
        # Decode batch
        # ----------------------------------------------------

        captions_cpu = captions.cpu().tolist()

        for i, image_index in enumerate(
            batch_indices
        ):

            generated_captions[
                int(image_index)
            ] = decode_caption(
                captions_cpu[i]
            )

    return generated_captions


# ============================================================
# RUN GENERATION
# ============================================================

generated_captions = generate_captions_greedy(
    unique_test_images,
    batch_size=128,
    max_length=MAX_LENGTH
)

print()
print("=" * 60)
print("CAPTION GENERATION COMPLETE")
print("=" * 60)

print(
    "Generated captions:",
    len(generated_captions)
)

assert len(generated_captions) == len(
    unique_test_images
)

print("✓ All test images generated")

Generating test captions:   0%|          | 0/94 [00:00<?, ?it/s]


CAPTION GENERATION COMPLETE
Generated captions: 11987
✓ All test images generated


STEP 16 — Decode Ground-Truth Captions

In [ ]:
# ============================================================
# STEP 16 — DECODE GROUND-TRUTH REFERENCES
# ============================================================

decoded_references = {}

for image_index, captions in test_references.items():

    decoded_references[image_index] = [
        decode_caption(caption)
        for caption in captions
    ]

print("=" * 60)
print("REFERENCE CAPTIONS DECODED")
print("=" * 60)

print(
    "Images:",
    len(decoded_references)
)

# Show one example
example_image = unique_test_images[0]

print()
print("Example image index:", example_image)

print("Generated:")
print(generated_captions[int(example_image)])

print()
print("References:")

for ref in decoded_references[
    int(example_image)
]:
    print("-", ref)

print()
print("✓ Ground-truth captions decoded")

REFERENCE CAPTIONS DECODED
Images: 11987

Example image index: 0
Generated:
A little girl is standing on a wooden staircase

References:
- A child in a pink dress is climbing up a set of stairs in an entry way
- A girl going into a wooden building
- A little girl climbing into a wooden playhouse
- A little girl climbing the stairs to her playhouse
- A little girl in a pink dress going into a wooden cabin

✓ Ground-truth captions decoded


STEP 17 — Install Evaluation Metrics

In [ ]:
# ============================================================
# STEP 17 — INSTALL EVALUATION LIBRARIES
# ============================================================

!pip -q install nltk rouge-score pycocoevalcap

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 10.2 MB/s eta 0:00:00


STEP 18 — Calculate BLEU + METEOR + ROUGE-L

In [ ]:
# ============================================================
# STEP 18 — BLEU / METEOR / ROUGE-L
# ============================================================

import nltk

nltk.download(
    "wordnet",
    quiet=True
)

nltk.download(
    "omw-1.4",
    quiet=True
)

from nltk.translate.bleu_score import (
    corpus_bleu,
    SmoothingFunction
)

from nltk.translate.meteor_score import (
    meteor_score
)

from rouge_score import rouge_scorer


# ------------------------------------------------------------
# Prepare tokenized captions
# ------------------------------------------------------------

references_tokenized = []
hypotheses_tokenized = []

meteor_scores = []

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = []


for image_index in unique_test_images:

    image_index = int(image_index)

    refs = decoded_references[
        image_index
    ]

    hypothesis = generated_captions[
        image_index
    ]

    refs_tokens = [
        ref.split()
        for ref in refs
    ]

    hyp_tokens = hypothesis.split()

    references_tokenized.append(
        refs_tokens
    )

    hypotheses_tokenized.append(
        hyp_tokens
    )

    # METEOR
    meteor_scores.append(
        meteor_score(
            refs_tokens,
            hyp_tokens
        )
    )

    # ROUGE-L
    best_rouge = 0.0

    for ref in refs:

        score = rouge.score(
            ref,
            hypothesis
        )["rougeL"].fmeasure

        best_rouge = max(
            best_rouge,
            score
        )

    rouge_scores.append(
        best_rouge
    )


# ------------------------------------------------------------
# BLEU
# ------------------------------------------------------------

smooth = SmoothingFunction().method4

bleu1 = corpus_bleu(
    references_tokenized,
    hypotheses_tokenized,
    weights=(1, 0, 0, 0),
    smoothing_function=smooth
)

bleu2 = corpus_bleu(
    references_tokenized,
    hypotheses_tokenized,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smooth
)

bleu3 = corpus_bleu(
    references_tokenized,
    hypotheses_tokenized,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smooth
)

bleu4 = corpus_bleu(
    references_tokenized,
    hypotheses_tokenized,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smooth
)

meteor = float(
    np.mean(meteor_scores)
)

rouge_l = float(
    np.mean(rouge_scores)
)

print("=" * 60)
print("MODEL 1 — TEXT METRICS")
print("=" * 60)

print(f"BLEU-1  : {bleu1:.4f}")
print(f"BLEU-2  : {bleu2:.4f}")
print(f"BLEU-3  : {bleu3:.4f}")
print(f"BLEU-4  : {bleu4:.4f}")
print(f"METEOR  : {meteor:.4f}")
print(f"ROUGE-L : {rouge_l:.4f}")

MODEL 1 — TEXT METRICS
BLEU-1  : 0.7201
BLEU-2  : 0.5376
BLEU-3  : 0.3883
BLEU-4  : 0.2780
METEOR  : 0.4802
ROUGE-L : 0.5342


STEP 19 — CIDEr

In [ ]:
# ============================================================
# STEP 19 — CIDEr
# ============================================================

from pycocoevalcap.cider.cider import Cider


gts = {}
res = {}

for image_index in unique_test_images:

    image_index = int(image_index)

    # CIDEr expects lists of strings
    gts[image_index] = decoded_references[
        image_index
    ]

    res[image_index] = [
        generated_captions[
            image_index
        ]
    ]


cider_scorer = Cider()

cider_score, cider_scores = (
    cider_scorer.compute_score(
        gts,
        res
    )
)

print("=" * 60)
print("CIDEr")
print("=" * 60)

print(
    f"CIDEr : {cider_score:.4f}"
)

CIDEr
CIDEr : 0.8631


STEP 20 — Final Model-1 Results

In [ ]:
# ============================================================
# STEP 20 — FINAL MODEL 1 RESULTS
# ============================================================

model_1_results = {

    "model": "CLIP + LSTM Optimized",

    "best_validation_loss": float(
        checkpoint["best_val_loss"]
    ),

    "BLEU-1": float(bleu1),
    "BLEU-2": float(bleu2),
    "BLEU-3": float(bleu3),
    "BLEU-4": float(bleu4),

    "METEOR": float(meteor),

    "ROUGE-L": float(rouge_l),

    "CIDEr": float(cider_score),

    "test_images": int(
        len(unique_test_images)
    ),

    "training_epochs": int(
        checkpoint["epoch"]
    )
}

print("=" * 70)
print("MODEL 1 — FINAL RESULTS")
print("=" * 70)

for metric, value in model_1_results.items():

    if isinstance(value, float):
        print(
            f"{metric:<25}: {value:.4f}"
        )

    else:
        print(
            f"{metric:<25}: {value}"
        )

MODEL 1 — FINAL RESULTS
model                    : CLIP + LSTM Optimized
best_validation_loss     : 2.9457
BLEU-1                   : 0.7201
BLEU-2                   : 0.5376
BLEU-3                   : 0.3883
BLEU-4                   : 0.2780
METEOR                   : 0.4802
ROUGE-L                  : 0.5342
CIDEr                    : 0.8631
test_images              : 11987
training_epochs          : 10


STEP 21 — Save Results to Drive

In [ ]:
# ============================================================
# STEP 21 — SAVE MODEL 1 RESULTS
# ============================================================

RESULTS_PATH = (
    MODEL_DIR
    / "evaluation_results.json"
)

GENERATED_PATH = (
    MODEL_DIR
    / "generated_test_captions.json"
)

# Save metrics
with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_1_results,
        f,
        indent=4
    )

# Save generated captions
with open(
    GENERATED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            str(k): v
            for k, v in generated_captions.items()
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 60)
print("MODEL 1 RESULTS SAVED")
print("=" * 60)

print("Metrics   :", RESULTS_PATH)
print("Captions  :", GENERATED_PATH)

print()
print("✓ Model 1 is now fully evaluated")

MODEL 1 RESULTS SAVED
Metrics   : /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/evaluation_results.json
Captions  : /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/generated_test_captions.json

✓ Model 1 is now fully evaluated


STEP 22 — Show 10 Qualitative Examples

In [ ]:
# ============================================================
# STEP 22 — QUALITATIVE CAPTION EXAMPLES
# ============================================================

print("=" * 70)
print("MODEL 1 — GENERATED CAPTION EXAMPLES")
print("=" * 70)

for n, image_index in enumerate(
    unique_test_images[:10],
    start=1
):

    image_index = int(image_index)

    print()
    print("-" * 70)
    print(f"Example {n}")
    print(f"Image index: {image_index}")

    print()
    print("GENERATED:")
    print(
        generated_captions[
            image_index
        ]
    )

    print()
    print("GROUND TRUTH:")

    for ref in decoded_references[
        image_index
    ]:

        print(
            "•",
            ref
        )# ============================================================
# GENERATE CAPTIONS FOR THE COMPLETE TEST SET
# ============================================================

import torch
from tqdm.auto import tqdm

model.eval()

all_predictions = []
all_references = []
all_image_ids = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Generating captions"):

        # ----------------------------------------------------
        # Get images
        # ----------------------------------------------------
        images = batch[0].to(device)

        # ----------------------------------------------------
        # Generate caption using YOUR EXISTING beam-search
        # function
        # ----------------------------------------------------
        for i in range(images.size(0)):

            image = images[i].unsqueeze(0)

            # Use the same function that you already used
            # successfully for your example captions.
            caption = generate_caption(
                image,
                encoder,
                decoder,
                vocab,
                device=device,
                beam_width=5,
                max_length=20
            )

            all_predictions.append(caption)

            # ------------------------------------------------
            # References
            # ------------------------------------------------
            if len(batch) > 2:
                refs = batch[2][i]
            else:
                refs = batch[1][i]

            if isinstance(refs, str):
                refs = [refs]

            all_references.append(refs)

            # ------------------------------------------------
            # Image ID
            # ------------------------------------------------
            if len(batch) > 3:
                all_image_ids.append(batch[3][i])
            else:
                all_image_ids.append(i)


print("\n========================================")
print("CAPTION GENERATION COMPLETE")
print("========================================")
print("Images processed :", len(all_predictions))
print("Predictions      :", len(all_predictions))
print("References       :", len(all_references))

print("\nSample results:\n")

for i in range(min(10, len(all_predictions))):
    print(f"Image {i}")
    print("Generated :", all_predictions[i])
    print("Reference :", all_references[i])
    print("-" * 70)

MODEL 1 — GENERATED CAPTION EXAMPLES

----------------------------------------------------------------------
Example 1
Image index: 0

GENERATED:
A little girl is standing on a wooden staircase

GROUND TRUTH:
• A child in a pink dress is climbing up a set of stairs in an entry way
• A girl going into a wooden building
• A little girl climbing into a wooden playhouse
• A little girl climbing the stairs to her playhouse
• A little girl in a pink dress going into a wooden cabin

----------------------------------------------------------------------
Example 2
Image index: 23

GENERATED:
A man and a child are rowing a canoe in a lake

GROUND TRUTH:
• A man and a baby are in a yellow kayak on water
• A man and a little boy in blue life jackets are rowing a yellow canoe
• A man and child kayak through gentle waters
• A man and young boy ride in a yellow kayak
• Man and child in yellow kayak

----------------------------------------------------------------------
Example 3
Image index: 31

GENE

In [ ]:
# ============================================================
# RESTORE CORRECT CLIP + LSTM INFERENCE FUNCTION
# MATCHES YOUR ACTUAL OptimizedCLIPLSTM MODEL
# ============================================================

import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Vocabulary lookup
# ------------------------------------------------------------

ID_TO_WORD = {
    int(k): v
    for k, v in id_to_word.items()
}


def ids_to_caption(token_ids):

    words = []

    for token_id in token_ids:

        token_id = int(token_id)

        if token_id == START_ID or token_id == PAD_ID:
            continue

        if token_id == END_ID:
            break

        word = ID_TO_WORD.get(
            token_id,
            "<UNK>"
        )

        if word not in {
            "<PAD>",
            "<start>",
            "<end>"
        }:
            words.append(word)

    return " ".join(words).strip()


# ============================================================
# CORRECT AUTOREGRESSIVE GENERATOR
# ============================================================

@torch.no_grad()
def generate_greedy_batch(
    image_features,
    max_length=MAX_LENGTH
):

    model.eval()

    image_features = image_features.to(
        DEVICE,
        non_blocking=True
    ).float()

    batch_size = image_features.size(0)

    # --------------------------------------------------------
    # YOUR ACTUAL MODEL:
    #
    # self.image_projection(...)
    # self.cell_projection(...)
    # --------------------------------------------------------

    h = model.image_projection(
        image_features
    ).unsqueeze(0).repeat(
        model.num_layers,
        1,
        1
    )

    c = model.cell_projection(
        image_features
    ).unsqueeze(0).repeat(
        model.num_layers,
        1,
        1
    )

    # --------------------------------------------------------
    # Output sequence
    # --------------------------------------------------------

    sequences = torch.full(
        (
            batch_size,
            max_length
        ),
        PAD_ID,
        dtype=torch.long,
        device=DEVICE
    )

    sequences[:, 0] = START_ID

    current = torch.full(
        (batch_size,),
        START_ID,
        dtype=torch.long,
        device=DEVICE
    )

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=DEVICE
    )

    logprob_sum = torch.zeros(
        batch_size,
        dtype=torch.float32,
        device=DEVICE
    )

    token_count = torch.zeros(
        batch_size,
        dtype=torch.float32,
        device=DEVICE
    )

    # --------------------------------------------------------
    # AUTOREGRESSIVE GENERATION
    # --------------------------------------------------------

    for t in range(
        1,
        max_length
    ):

        embedding = model.embedding(
            current
        ).unsqueeze(1)

        lstm_output, (h, c) = model.lstm(
            embedding,
            (h, c)
        )

        logits = model.output_layer(
            lstm_output.squeeze(1)
        )

        # Never generate PAD or START
        logits[:, PAD_ID] = -float("inf")
        logits[:, START_ID] = -float("inf")

        log_probs = F.log_softmax(
            logits,
            dim=-1
        )

        next_token = torch.argmax(
            log_probs,
            dim=-1
        )

        chosen_logprob = (
            log_probs
            .gather(
                1,
                next_token.unsqueeze(1)
            )
            .squeeze(1)
        )

        # Keep finished sequences at END
        next_token = torch.where(
            finished,
            torch.full_like(
                next_token,
                END_ID
            ),
            next_token
        )

        active = ~finished

        sequences[:, t] = next_token

        logprob_sum[active] += (
            chosen_logprob[active].float()
        )

        token_count[active] += 1.0

        finished = (
            finished |
            next_token.eq(END_ID)
        )

        current = next_token

        if bool(finished.all()):
            break

    mean_token_probability = torch.exp(
        logprob_sum /
        token_count.clamp_min(1.0)
    )

    return (
        sequences.cpu(),
        mean_token_probability.cpu()
    )


# ============================================================
# TEST WITH 2 IMAGES ONLY
# ============================================================

print("=" * 70)
print("TESTING CORRECT INFERENCE FUNCTION")
print("=" * 70)

test_image_indices = np.unique(
    test_indices
)[:2]

test_features = torch.from_numpy(
    clip_feature_matrix[
        test_image_indices
    ]
)

test_ids, test_conf = generate_greedy_batch(
    test_features,
    max_length=MAX_LENGTH
)

for i in range(
    len(test_ids)
):

    print()
    print(
        f"Sample {i + 1}:",
        ids_to_caption(
            test_ids[i].tolist()
        )
    )

    print(
        "Confidence:",
        f"{test_conf[i].item():.4f}"
    )

print()
print("✓ CORRECT INFERENCE FUNCTION WORKING")

TESTING CORRECT INFERENCE FUNCTION

Sample 1: A little girl in a pink dress is standing in front of a wooden fence
Confidence: 0.4504

Sample 2: A man and a child are in a canoe
Confidence: 0.4326

✓ CORRECT INFERENCE FUNCTION WORKING


In [ ]:
# ============================================================
# MODEL 1 — GENERATE ALL TEST CAPTIONS
# ============================================================

import json
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64

RESULTS_DIR = (
    MODEL_DIR / "Evaluation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTIONS_PATH = (
    RESULTS_DIR /
    "clip_lstm_test_predictions.json"
)

# ------------------------------------------------------------
# Unique test images
# ------------------------------------------------------------

unique_test_indices = np.unique(
    test_indices
)

print("=" * 70)
print("GENERATING FULL TEST SET")
print("=" * 70)

print(
    "Unique test images:",
    len(unique_test_indices)
)

print(
    "Batch size:",
    EVAL_BATCH_SIZE
)

# ------------------------------------------------------------
# Resume support
# ------------------------------------------------------------

predictions = {}

if PREDICTIONS_PATH.exists():

    print()
    print("Existing prediction file found.")
    print("Loading previous progress...")

    with open(
        PREDICTIONS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        predictions = json.load(f)

    print(
        "Already generated:",
        len(predictions)
    )

# ------------------------------------------------------------
# Only process images not already generated
# ------------------------------------------------------------

remaining_indices = [
    int(index)
    for index in unique_test_indices
    if str(int(index)) not in predictions
]

print(
    "Remaining:",
    len(remaining_indices)
)

# ------------------------------------------------------------
# Generate
# ------------------------------------------------------------

model.eval()

for start in tqdm(
    range(
        0,
        len(remaining_indices),
        EVAL_BATCH_SIZE
    ),
    desc="Generating captions"
):

    batch_indices = remaining_indices[
        start:
        start + EVAL_BATCH_SIZE
    ]

    batch_features = torch.from_numpy(
        clip_feature_matrix[
            batch_indices
        ]
    )

    batch_ids, batch_conf = (
        generate_greedy_batch(
            batch_features,
            max_length=MAX_LENGTH
        )
    )

    for i, image_index in enumerate(
        batch_indices
    ):

        prediction = ids_to_caption(
            batch_ids[i].tolist()
        )

        predictions[str(image_index)] = {
            "prediction": prediction,
            "mean_token_probability": float(
                batch_conf[i].item()
            )
        }

    # Save every 10 batches
    batch_number = (
        start // EVAL_BATCH_SIZE
    ) + 1

    if batch_number % 10 == 0:

        with open(
            PREDICTIONS_PATH,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                predictions,
                f,
                indent=2,
                ensure_ascii=False
            )

# ------------------------------------------------------------
# Final save
# ------------------------------------------------------------

with open(
    PREDICTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print("=" * 70)
print("FULL TEST GENERATION COMPLETE")
print("=" * 70)

print(
    "Generated:",
    len(predictions)
)

print(
    "Expected:",
    len(unique_test_indices)
)

print(
    "Saved:",
    PREDICTIONS_PATH
)

assert len(predictions) == len(
    unique_test_indices
)

print()
print("✓ ALL TEST CAPTIONS GENERATED")

GENERATING FULL TEST SET
Unique test images: 11987
Batch size: 64
Remaining: 11987


Generating captions:   0%|          | 0/188 [00:00<?, ?it/s]


FULL TEST GENERATION COMPLETE
Generated: 11987
Expected: 11987
Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/Evaluation/clip_lstm_test_predictions.json

✓ ALL TEST CAPTIONS GENERATED


In [ ]:
# ============================================================
# MODEL 1 — BUILD REFERENCE CAPTIONS
# ============================================================

print("=" * 70)
print("BUILDING TEST REFERENCES")
print("=" * 70)

evaluation_data = []

for image_index in unique_test_indices:

    image_index = int(image_index)

    # All captions belonging to this image
    mask = (
        test_indices == image_index
    )

    reference_ids = test_captions[
        mask
    ]

    references = [
        ids_to_caption(
            caption.tolist()
        )
        for caption in reference_ids
    ]

    references = [
        ref.strip()
        for ref in references
        if ref.strip()
    ]

    prediction_data = predictions[
        str(image_index)
    ]

    prediction = prediction_data[
        "prediction"
    ]

    confidence = prediction_data[
        "mean_token_probability"
    ]

    evaluation_data.append({

        "image_index":
            image_index,

        "prediction":
            prediction,

        "confidence":
            confidence,

        "references":
            references
    })


print()
print(
    "Evaluation images:",
    len(evaluation_data)
)

print(
    "References per image:",
    len(
        evaluation_data[0]["references"]
    )
)

print()
print("Example")
print("-" * 60)

print(
    "Prediction:",
    evaluation_data[0]["prediction"]
)

for i, ref in enumerate(
    evaluation_data[0]["references"],
    1
):

    print(
        f"Reference {i}:",
        ref
    )

print()
print("✓ Evaluation data prepared")

BUILDING TEST REFERENCES

Evaluation images: 11987
References per image: 5

Example
------------------------------------------------------------
Prediction: A little girl in a pink dress is standing in front of a wooden fence
Reference 1: A child in a pink dress is climbing up a set of stairs in an entry way
Reference 2: A girl going into a wooden building
Reference 3: A little girl climbing into a wooden playhouse
Reference 4: A little girl climbing the stairs to her playhouse
Reference 5: A little girl in a pink dress going into a wooden cabin

✓ Evaluation data prepared


In [ ]:
# ============================================================
# MODEL 1 — BLEU / METEOR / ROUGE-L
# ============================================================

import nltk

nltk.download(
    "wordnet",
    quiet=True
)

nltk.download(
    "omw-1.4",
    quiet=True
)

from nltk.translate.bleu_score import (
    corpus_bleu,
    SmoothingFunction
)

from nltk.translate.meteor_score import (
    meteor_score
)

# ------------------------------------------------------------
# Prepare tokens
# ------------------------------------------------------------

references_tokenized = []
predictions_tokenized = []

meteor_values = []

for item in evaluation_data:

    refs = [
        ref.lower().split()
        for ref in item["references"]
    ]

    prediction = (
        item["prediction"]
        .lower()
        .split()
    )

    references_tokenized.append(
        refs
    )

    predictions_tokenized.append(
        prediction
    )

    # METEOR uses individual samples
    if prediction and refs:

        meteor_values.append(
            meteor_score(
                refs,
                prediction
            )
        )


# ------------------------------------------------------------
# BLEU
# ------------------------------------------------------------

smooth = (
    SmoothingFunction()
    .method1
)

bleu1 = corpus_bleu(
    references_tokenized,
    predictions_tokenized,
    weights=(1, 0, 0, 0),
    smoothing_function=smooth
)

bleu2 = corpus_bleu(
    references_tokenized,
    predictions_tokenized,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smooth
)

bleu3 = corpus_bleu(
    references_tokenized,
    predictions_tokenized,
    weights=(
        1/3,
        1/3,
        1/3,
        0
    ),
    smoothing_function=smooth
)

bleu4 = corpus_bleu(
    references_tokenized,
    predictions_tokenized,
    weights=(
        0.25,
        0.25,
        0.25,
        0.25
    ),
    smoothing_function=smooth
)


# ------------------------------------------------------------
# METEOR
# ------------------------------------------------------------

meteor_value = float(
    np.mean(meteor_values)
)


# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

def lcs_length(a, b):

    if not a or not b:
        return 0

    previous = [
        0
    ] * (
        len(b) + 1
    )

    for token_a in a:

        current = [
            0
        ] * (
            len(b) + 1
        )

        for j, token_b in enumerate(
            b,
            start=1
        ):

            if token_a == token_b:

                current[j] = (
                    previous[j - 1] + 1
                )

            else:

                current[j] = max(
                    previous[j],
                    current[j - 1]
                )

        previous = current

    return previous[-1]


def rouge_l_f1(
    prediction,
    reference
):

    lcs = lcs_length(
        prediction,
        reference
    )

    if lcs == 0:
        return 0.0

    precision = (
        lcs / len(prediction)
    )

    recall = (
        lcs / len(reference)
    )

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


rouge_values = []

for refs, prediction in zip(
    references_tokenized,
    predictions_tokenized
):

    scores = [
        rouge_l_f1(
            prediction,
            ref
        )
        for ref in refs
        if ref
    ]

    if scores:

        # Best matching reference
        rouge_values.append(
            max(scores)
        )


rouge_l = float(
    np.mean(rouge_values)
)


# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("=" * 70)
print("MODEL 1 — TEXT METRICS")
print("=" * 70)

print(
    f"BLEU-1  : {bleu1:.4f}"
)

print(
    f"BLEU-2  : {bleu2:.4f}"
)

print(
    f"BLEU-3  : {bleu3:.4f}"
)

print(
    f"BLEU-4  : {bleu4:.4f}"
)

print(
    f"METEOR  : {meteor_value:.4f}"
)

print(
    f"ROUGE-L : {rouge_l:.4f}"
)

MODEL 1 — TEXT METRICS
BLEU-1  : 0.6197
BLEU-2  : 0.4401
BLEU-3  : 0.3018
BLEU-4  : 0.2061
METEOR  : 0.4253
ROUGE-L : 0.4608


STEP 4 — CIDEr

In [ ]:
# ============================================================
# MODEL 1 — INSTALL CIDEr
# ============================================================

!pip -q install pycocoevalcap

# ============================================================
# MODEL 1 — CIDEr
# ============================================================

from pycocoevalcap.cider.cider import Cider

gts = {}
res = {}

for item in evaluation_data:

    image_id = str(
        item["image_index"]
    )

    gts[image_id] = (
        item["references"]
    )

    res[image_id] = [
        item["prediction"]
    ]


cider_scorer = Cider()

cider_score, individual_scores = (
    cider_scorer.compute_score(
        gts,
        res
    )
)

print("=" * 70)
print("CIDEr")
print("=" * 70)

print(
    f"CIDEr : {cider_score:.4f}"
)

CIDEr
CIDEr : 0.6092


In [ ]:
# ============================================================
# MODEL 1 — FINAL RESULTS
# ============================================================

model_1_results = {

    "model":
        "Optimized CLIP + LSTM",

    "test_images":
        len(evaluation_data),

    "best_validation_loss":
        float(
            checkpoint[
                "best_val_loss"
            ]
        ),

    "BLEU-1":
        float(bleu1),

    "BLEU-2":
        float(bleu2),

    "BLEU-3":
        float(bleu3),

    "BLEU-4":
        float(bleu4),

    "METEOR":
        float(meteor_value),

    "ROUGE-L":
        float(rouge_l),

    "CIDEr":
        float(cider_score),

    "mean_generation_confidence":
        float(
            np.mean([
                item["confidence"]
                for item in evaluation_data
            ])
        )
}


FINAL_RESULTS_PATH = (
    RESULTS_DIR /
    "CLIP_LSTM_FINAL_RESULTS.json"
)

with open(
    FINAL_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_1_results,
        f,
        indent=4
    )


print()
print("=" * 70)
print("MODEL 1 — FINAL RESULTS")
print("=" * 70)

for key, value in model_1_results.items():

    if isinstance(
        value,
        float
    ):

        print(
            f"{key:<30}: {value:.4f}"
        )

    else:

        print(
            f"{key:<30}: {value}"
        )

print()
print(
    "Saved:",
    FINAL_RESULTS_PATH
)

print()
print(
    "✓ MODEL 1 COMPLETE"
)


MODEL 1 — FINAL RESULTS
model                         : Optimized CLIP + LSTM
test_images                   : 11987
best_validation_loss          : 2.9457
BLEU-1                        : 0.6197
BLEU-2                        : 0.4401
BLEU-3                        : 0.3018
BLEU-4                        : 0.2061
METEOR                        : 0.4253
ROUGE-L                       : 0.4608
CIDEr                         : 0.6092
mean_generation_confidence    : 0.4132

Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Models/CLIP_LSTM_Optimized/Evaluation/CLIP_LSTM_FINAL_RESULTS.json

✓ MODEL 1 COMPLETE
